# Route Optimization Algorithm Analysis

This notebook provides a detailed analysis of the Floyd-Warshall algorithm compared to other path-finding algorithms like Dijkstra's algorithm and A* algorithm.

## Time Complexity Analysis

- **Floyd-Warshall**: O(V³) where V is the number of vertices
- **Dijkstra's Algorithm**: O(V²log V) for all pairs, O(E + V log V) for single source
- **A* Algorithm**: O(E) in the worst case, but typically much faster due to heuristics

In [ ]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath('../'))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import time

from algorithms.floyd_warshall import FloydWarshall, compare_with_networkx, analyze_time_complexity
from analysis.comparison import compare_algorithms, visualize_complexity, generate_random_graphs

## Generate Test Graphs

Let's generate some random graphs of different sizes to test our algorithms.

In [ ]:
# Generate random graphs of different sizes
sizes = [10, 20, 50, 100]
graphs = generate_random_graphs(sizes, density=0.3)

# Display basic information about the generated graphs
for size, G in graphs.items():
    print(f"Graph with {size} nodes: {len(G.edges())} edges, connected: {nx.is_connected(G)}")

## Algorithm Comparison on a Small Graph

Let's compare the performance of different algorithms on a small graph.

In [ ]:
# Use the smallest graph for detailed comparison
small_graph = graphs[10]

# Select source and target nodes
source = 0
target = 5  # Assuming the graph has at least 6 nodes

# Compare algorithms
results = compare_algorithms(small_graph, source, target)

# Display results
print("Algorithm Comparison Results:")
print(results[["algorithm", "distance", "time", "path_length"]])

# Visualize comparison
plt.figure(figsize=(10, 6))
ax = plt.gca()
visualize_comparison(results, ax)
plt.tight_layout()
plt.show()

## Path Visualization

Let's visualize the paths found by different algorithms.

In [ ]:
# Create a figure with subplots for each algorithm
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Get positions for nodes (consistent across subplots)
pos = nx.spring_layout(small_graph, seed=42)

# Plot each algorithm's path
for i, algo in enumerate(['floyd_warshall', 'dijkstra', 'astar']):
    ax = axes[i]
    path = results.loc[algo, 'path']
    
    # Draw the graph
    nx.draw_networkx_nodes(small_graph, pos, node_color='lightblue', node_size=300, ax=ax)
    nx.draw_networkx_edges(small_graph, pos, width=1.0, alpha=0.5, ax=ax)
    nx.draw_networkx_labels(small_graph, pos, font_size=10, ax=ax)
    
    # Highlight the path
    path_edges = [(path[i], path[i+1]) for i in range(len(path)-1)]
    nx.draw_networkx_nodes(small_graph, pos, nodelist=path, node_color='red', node_size=300, ax=ax)
    nx.draw_networkx_edges(small_graph, pos, edgelist=path_edges, width=2.5, edge_color='red', ax=ax)
    
    # Add title
    algo_name = results.loc[algo, 'algorithm']
    distance = results.loc[algo, 'distance']
    time_ms = results.loc[algo, 'time'] * 1000  # Convert to ms
    ax.set_title(f"{algo_name}
Distance: {distance:.2f}, Time: {time_ms:.2f} ms")
    ax.axis('off')

plt.tight_layout()
plt.show()

## Time Complexity Analysis

Let's analyze how the execution time of each algorithm scales with graph size.

In [ ]:
# Analyze time complexity
sizes = [10, 20, 30, 40, 50]
results = analyze_time_complexity(sizes)

# Display results
print("Time Complexity Analysis Results:")
for size, fw_time, dijkstra_time, bf_time in zip(
    results['sizes'], 
    results['floyd_warshall'], 
    results['dijkstra_all'], 
    results['bellman_ford_all']
):
    print(f"Size {size}: Floyd-Warshall {fw_time:.6f}s, Dijkstra {dijkstra_time:.6f}s, Bellman-Ford {bf_time:.6f}s")

# Visualize complexity
plt.figure(figsize=(10, 6))
ax = plt.gca()
visualize_complexity(results['sizes'], results, ax)
plt.tight_layout()
plt.show()

## Theoretical vs. Empirical Time Complexity

Let's compare the theoretical time complexity with our empirical measurements.

In [ ]:
# Create arrays for theoretical complexity curves
n_values = np.array(results['sizes'])

# Normalize to match the first measured point
v3_values = n_values**3 / n_values[0]**3 * results['floyd_warshall'][0]  # O(V^3) for Floyd-Warshall
v2logv_values = n_values**2 * np.log(n_values) / (n_values[0]**2 * np.log(n_values[0])) * results['dijkstra_all'][0]  # O(V^2 log V) for Dijkstra all pairs
v3_values_bf = n_values**3 / n_values[0]**3 * results['bellman_ford_all'][0]  # O(V^3) for Bellman-Ford all pairs

# Plot measured vs theoretical
plt.figure(figsize=(12, 8))

# Floyd-Warshall
plt.subplot(1, 3, 1)
plt.plot(n_values, results['floyd_warshall'], 'o-', label='Measured')
plt.plot(n_values, v3_values, '--', label='Theoretical O(V³)')
plt.title('Floyd-Warshall')
plt.xlabel('Graph Size (nodes)')
plt.ylabel('Time (seconds)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# Dijkstra all pairs
plt.subplot(1, 3, 2)
plt.plot(n_values, results['dijkstra_all'], 'o-', label='Measured')
plt.plot(n_values, v2logv_values, '--', label='Theoretical O(V² log V)')
plt.title('Dijkstra (All Pairs)')
plt.xlabel('Graph Size (nodes)')
plt.ylabel('Time (seconds)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# Bellman-Ford all pairs
plt.subplot(1, 3, 3)
plt.plot(n_values, results['bellman_ford_all'], 'o-', label='Measured')
plt.plot(n_values, v3_values_bf, '--', label='Theoretical O(V³)')
plt.title('Bellman-Ford (All Pairs)')
plt.xlabel('Graph Size (nodes)')
plt.ylabel('Time (seconds)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Conclusion

Based on our analysis, we can draw the following conclusions:

1. **Floyd-Warshall Algorithm**:
   - Time Complexity: O(V³)
   - Best for: Finding all-pairs shortest paths in dense graphs
   - Advantages: Simple implementation, works with negative edge weights (but not negative cycles)
   - Disadvantages: Cubic time complexity makes it inefficient for very large graphs

2. **Dijkstra's Algorithm**:
   - Time Complexity: O(E + V log V) for single source, O(V² log V) for all pairs
   - Best for: Finding single-source shortest paths in graphs with non-negative edge weights
   - Advantages: Efficient for sparse graphs, can be optimized with priority queues
   - Disadvantages: Doesn't work with negative edge weights

3. **A* Algorithm**:
   - Time Complexity: O(E) in worst case, but typically much better due to heuristics
   - Best for: Finding single-source shortest paths with additional heuristic information
   - Advantages: Often faster than Dijkstra's algorithm due to heuristic guidance
   - Disadvantages: Requires a good heuristic function, doesn't work with negative edge weights

For our route optimization system, the Floyd-Warshall algorithm is a good choice when:
- The graph is relatively small (up to a few hundred nodes)
- We need to compute all-pairs shortest paths
- The graph may have negative edge weights (but no negative cycles)

For larger graphs or when only specific paths are needed, Dijkstra's or A* algorithms may be more efficient.